In [ ]:
cd ../..

## Feature Ideas

- `week_of_year`
- `month_of_year`
- `day_of_month`
- `day_of_week`
- `customer_age`
- `fn_not_null`: Whether the FN customer field is null
- `customer_is_active`
- `club_member_status`
- `fashion_news_frequency`
- Customer aggs full history:
    - `total_purchase_count`
    - `total_purchase_amount`
    - `avg_purchase_price`
    - `total_unique_articles_purchased`
    - `prop_sales_channel_2`
    - `modal_dept_no`
    - `modal_section_no`
    - `modal_color_id`
- For each of the past 5 weeks:
    - `num_purchases`
    - `purchased_amount`
    - `avg_purchase_price`
    - `num_unique_articles_purchased`
    - `prop_sales_channel_2`
    - Aggs over items customer purchased in that period:
        - `avg_monthly_sales_amount`
        - `avg_monthly_sales_count`
        - `avg_days_since_last_sale`
        - `modal_dept_no`
        - `modal_section_no`
        - `modal_color_id`

In [ ]:
import duckdb
from torch_frame.utils import infer_df_stype

import utils

conn = duckdb.connect('hm/hm.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

In [ ]:
with open('hm/user-churn/feats.sql', 'r') as f:
    # run once with train_labels and once with val_labels
    template = f.read()

# create train, val and test features
# takes 1 - 5 mins
for s in ['train', 'val', 'test']:
    print(f'Creating {s} table')
    query = utils.render_jinja_sql(template, dict(set=s, subsample=0))
    conn.sql(query)
    print(f'{s} table created')

In [ ]:
utils.validate_feature_tables('user_churn', conn)

In [ ]:
%%sql train_df <<
from user_churn_train_feats

In [ ]:
infer_df_stype(train_df)

In [ ]:
utils.feature_summary_df(train_df.sample(20_000), 'churn', classification=True)

## Feature Importances

In [ ]:
import numpy as np
import shap
from torch_frame import TaskType, stype
from torch_frame.data import Dataset
from torch_frame.gbdt import LightGBM

from inferred_stypes import task_to_stypes
from train_gbdt import TASK_PARAMS

TASK = 'rel-hm-user-churn'

task_params = TASK_PARAMS[TASK]

In [ ]:
%%sql val_df <<
select * from user_churn_val_feats;

In [ ]:
val_tf = Dataset(
    val_df,
    col_to_stype=task_to_stypes[TASK],
    target_col=task_params['target_col'],
).materialize().tensor_frame

In [ ]:
gbdt = LightGBM(task_type=task_params['task_type'])
gbdt.load(f'models/{TASK}_lgbm.json')
pred = gbdt.predict(tf_test=val_tf).numpy()

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

sample = np.random.randint(0, len(val_tf), size=10_000)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[sample])
shap_values = explainer.shap_values(val_arr, pred[sample])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=30, feature_names=feat_names)

In [ ]:
conn.close()